# Medical Q&A with Llama 2 and Pinecone on Google Colab

Runs the full app on Colab's free T4 GPU: PDF ingestion into Pinecone, cross-encoder reranking, and a quantized Llama 2 7B chat model answering through a LangChain RetrievalQA chain, served by Flask.

**Before you start**

1. **Runtime > Change runtime type > T4 GPU**, then Save.
2. Click the **key icon** in the left sidebar (Secrets), add a secret named `PINECONE_API_KEY` with your Pinecone key, and turn on **Notebook access**.
3. **Runtime > Run all**. The first run takes about 10 minutes (installing packages and downloading the 4 GB model).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU found. Use Runtime > Change runtime type > T4 GPU."

## 2. Get the project code

The notebook clones the GitHub repo. If the repo is not public yet, it asks you to upload `medical-qa-llama2-pinecone.zip` instead.

In [ ]:
import os, subprocess, zipfile, glob

REPO_URL = "https://github.com/harmanJLD/medical-qa-llama2-pinecone.git"
PROJECT = "/content/medical-qa-llama2-pinecone"

os.environ["GIT_TERMINAL_PROMPT"] = "0"
if not os.path.isdir(PROJECT):
    cloned = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT], capture_output=True, text=True)
    if cloned.returncode != 0:
        print("Could not clone the repo, so upload medical-qa-llama2-pinecone.zip instead.")
        from google.colab import files
        uploaded = files.upload()
        name = next(iter(uploaded))
        with zipfile.ZipFile(name) as z:
            z.extractall("/content")
        if not os.path.isdir(PROJECT):  # zip had a different top-level folder name
            found = [os.path.dirname(p) for p in glob.glob("/content/*/app.py")]
            os.rename(found[0], PROJECT)
%cd {PROJECT}
!ls

## 3. Install packages

Installs llama-cpp-python with CUDA support. It tries a prebuilt GPU wheel first, which takes about a minute. If that does not give GPU support, it builds from source, which takes about 10 to 15 minutes.

In [ ]:
!grep -v -i "llama-cpp-python" requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt
!pip install -q "llama-cpp-python==0.3.4" --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [ ]:
import subprocess, sys

def gpu_ok():
    check = "import llama_cpp; print(llama_cpp.llama_supports_gpu_offload())"
    out = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
    return out.stdout.strip() == "True"

if gpu_ok():
    print("llama-cpp-python has GPU support.")
else:
    print("Prebuilt wheel has no GPU support. Building from source with CUDA (10 to 15 minutes)...")
    !CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install -q --force-reinstall --no-cache-dir llama-cpp-python
    print("GPU support:", gpu_ok())

## 4. Configure keys and settings

In [ ]:
from google.colab import userdata

try:
    pinecone_key = userdata.get("PINECONE_API_KEY")
except Exception:
    pinecone_key = None
if not pinecone_key:
    raise SystemExit("Add a Colab secret named PINECONE_API_KEY (key icon in the left sidebar) and enable notebook access.")

settings = {
    "PINECONE_API_KEY": pinecone_key,
    "PINECONE_INDEX": "medical-qa",
    "LLAMA_MODEL_PATH": "models/llama-2-7b-chat.Q4_K_M.gguf",
    "LLAMA_GPU_LAYERS": "-1",   # all layers on the T4 GPU
    "LLAMA_N_CTX": "4096",
    "TOP_K": "4",
    "PORT": "5050",
}
with open(".env", "w") as f:
    f.write("\n".join(f"{k}={v}" for k, v in settings.items()) + "\n")
print("Wrote .env (key hidden).")

## 5. Download the model and a sample PDF

In [ ]:
import os
from huggingface_hub import hf_hub_download

os.makedirs("models", exist_ok=True)
path = hf_hub_download("TheBloke/Llama-2-7B-Chat-GGUF", "llama-2-7b-chat.Q4_K_M.gguf", local_dir="models")
print("Model:", path, f"{os.path.getsize(path) / 1e9:.2f} GB")

os.makedirs("data", exist_ok=True)
pdf = "data/4-steps-manage-diabetes.pdf"
if not os.path.exists(pdf):
    !curl -sL -o {pdf} "https://stacks.cdc.gov/view/cdc/55506/cdc_55506_DS1.pdf"
!ls -lh data

## 6. Index the PDFs in Pinecone

Safe to re-run: vector IDs are stable, so existing vectors are updated, not duplicated. To add more documents, upload PDFs into the `data` folder (file browser on the left) and run this cell again.

In [ ]:
!python scripts/ingest.py

## 7. Run the tests

In [ ]:
!python -m pytest -q

## 8. Start the app

In [ ]:
import subprocess, time, urllib.request, json

!pkill -f "python app.py" || true
log = open("/content/app.log", "w")
server = subprocess.Popen(["python", "app.py"], stdout=log, stderr=subprocess.STDOUT)

for _ in range(180):
    time.sleep(2)
    if server.poll() is not None:
        break
    try:
        urllib.request.urlopen("http://127.0.0.1:5050/api/health", timeout=2)
        print("App is running.")
        break
    except Exception:
        pass
else:
    print("App did not start in time.")

!grep -E "Reranker|Loading Llama|Ready|Error|Traceback" /content/app.log | tail -n 15

## 9. Quick test from the notebook

In [ ]:
import json, time, urllib.request

def ask(question):
    req = urllib.request.Request(
        "http://127.0.0.1:5050/api/ask",
        data=json.dumps({"question": question}).encode(),
        headers={"Content-Type": "application/json"},
    )
    start = time.time()
    result = json.load(urllib.request.urlopen(req, timeout=300))
    print(f"Q: {question}\n")
    print(result["answer"], "\n")
    for s in result.get("sources", []):
        print(f"  - {s['source']}, p. {s['page']}")
    print(f"  ({time.time() - start:.1f} s)\n")

ask("What are the diabetes ABCs?")
ask("What causes vitamin D deficiency?")

## 10. Open the chat interface

**Option A (private):** a link through Colab that only works in your own browser while you are signed in to Google.

In [ ]:
from google.colab.output import eval_js
print("Open:", eval_js("google.colab.kernel.proxyPort(5050)"))

**Option B (shareable):** a temporary public link through a Cloudflare tunnel. Anyone with the link can use the app while this notebook runs, so share it only for demos.

In [ ]:
import re, subprocess, time

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
tunnel_log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:5050"],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(1)
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if found:
        url = found.group(0)
        break
print("Public link:", url or "not ready yet, run this cell again")

## 11. Logs and shutdown

In [ ]:
!tail -n 30 /content/app.log

In [ ]:
# Stop the app and the tunnel
!pkill -f "python app.py" || true
!pkill -f cloudflared || true
print("Stopped.")

## Optional: evaluation

Generates questions from the indexed pages with Llama 2 and measures retrieval hit@k, MRR, refusals for off-topic questions, citation accuracy, and latency. Stop the app first (cell above) to free GPU memory.

In [ ]:
!python eval/evaluate.py --generate 40 --answers 15